In [2]:
#| echo: false
#| output: false

# core dependencies
import pandas as pd
import numpy as np
import nflreadpy as nfl
import sys
import warnings

# Visualizations
import seaborn as sns
import matplotlib.pyplot as plt
import altair as alt

# Color accessibility library
import tol_colors as tc
from tol_colors import tol_cset

# Machine Learning
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

# routing help for importing data and helper functions
sys.path.append("..")

# helper functions
from helper_functions.clean_data import go_column, clean_fourth

# Warnings and display settings
pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore",category= UserWarning)
alt.data_transformers.enable("vegafusion")


DataTransformerRegistry.enable('vegafusion')

## 1. Introduction

In the NFL team's have three options on fourth down: punt, kick, or go for it. The way NFL teams approach fourth down decision making is dramatically shifting. Any sports talk radio show after a close game will doubtlessly have discourse about why their team should have been conservative and taken the points or been aggressive and gone for it. "Traditional" football fans swear by taking the points (i.e. attempting a field goal whenever given the opportunity) while "nerds" swear by black box statistical models urging teams to go for it more often. However, much of the discussions surrounding fourth down decision making is *reactionary*, it analyzes the result of the play devoid from the context surrounding the decision. What this article seeks to address are the circumstances that surround downs where NFL teams go for it, so fans of football can better understand  when their favorite NFL team will go for it, and grasp what factors went into that decision. Furthermore, this article will examine what factors drive success on fourth down attempts.

### Data Set(s) Used
All data used in this article is provided for via the nflverse, an API for interacting with a multitude of NFL data sources. In particular, NFL play by play data documenting every play from 2016 through 2025 was used in conjunction with Participation data from FTN. Participation data includes information such as the coverage a defense is in, the formation an offense is in, the route a receiver ran, etc. For a more detailed explanation to how the dataset used was acquired, check out the file load_data.py.

As stated previously, the raw NFL play by play dataset contains every single down that has occurred. What we are interested are plays that were fourth downs, and then being able to identify which of those downs teams went for it. This means that downs that were first, second, or third down must be filtered out. In addition, careful consideration must be taken to ensure the plays classified as a fourth down attempt are *genuine* attempts to convert them. This means excluding plays that were aborted, i.e. a punt or field goal attempt where a bad snap or other gaff caused a complete breakdown of the intended play. For detailed information on the filtering criteria used, check out the file clean_data.py

In [3]:
#| echo: false
#| warning: false

# Load in Main Data Set
data = pd.read_csv("../fourth.csv").drop(columns='Unnamed: 0')

# use go_column function to load add 
data = go_column(df=data)

data = clean_fourth(df=data)

# verify that fourth down attempts are correctly classified
#data.groupby("go")["play_type"].value_counts().reset_index()

/tmp/ipykernel_14883/1702693732.py:5: DtypeWarning: Columns (0: defense_man_zone_type, 1: defense_coverage_type) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("../fourth.csv").drop(columns='Unnamed: 0')


### Color Accessibility
Many color palettes used for data visualizations disregard the significant population of people who are color blind. When these unaccessible color schemes are used, the primary message a visualization is attempting to convey is lost. Therefore, this article will utilize the ***bright*** and ***muted*** color schemes developed by [Paul Tol](https://sronpersonalpages.nl/~pault/) for color blind accessibility.

In [4]:
#| echo: false
#| warning: false
cset = tc.bright
colors = [c for c in cset]  # convert to list

![](color_scheme.png "Title")

![](color_scheme_2.png "Title")

In [5]:
#| echo: false

cset_2 = tc.muted
colors_2 = [c for c in cset_2]  # convert to list

## 2. Methodology
The scope of this study is split into 4 different sections: 

### [When (and why) teams go for it](#sec-when)
Pertains to the game context surrounding downs when teams went for it.

### [Success Rate](#sec-success)
Measurement of fourth down attempt success rates across the league, and compared with attempt rate.

### [Event Analysis](#sec-event)
Analyzing runs and passes (separately) to understand what drives success for two types of plays.

### [Modeling Fourth Down Attempts](#sec-modeling) 
Leveraging interpretable machine learning predict when teams will go for it.

## 3. When (and why) teams go for it {#sec-when}

### Aggressiveness Over Time
To get a better contextual understanding of the league's fourth down approaches, we first will view how the entire league's aggressiveness as changed in the previous 10 years.

In [6]:
#| echo: false

# Get each seasons attempts and non attempts
rates = data.groupby(["season", "go"]).agg(
    n = ("go", "count")
).reset_index()

# pivot
rates = pd.pivot(data= rates, index= "season", columns= "go", values = "n" ).reset_index().rename(columns ={0.0: "no_go", 1.0: "go"} )
# pivot works by indexing on the column you want to represent an individual row, then column repressing the column you want to have a new column created for each distinct value in it, and values is what should go into that new column

rates = rates.assign(go_rate = round(rates["go"] / (rates["go"] + rates["no_go"]), 4))

In [7]:
#| echo: false

ag_chart = (alt.Chart(data= rates, title="Fourth Down Aggressiveness (2016-2025)")
        .mark_line(
            color='#4477AA', 
            point=alt.OverlayMarkDef(color="#EE6677", size=50, opacity = 0.5)
            )
        .encode(
            x=alt.X("season:N", title='Season', axis = alt.Axis(labelAngle=0)),
            y=alt.Y("go_rate:Q", title= "Go Rate"),
            tooltip=["season", "go_rate"]
        )
        .properties(width=600, height=300)
    ).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)

ag_chart

alt.Chart(...)

Since 2016, the fourth down go rate has nearly **doubled**, going from 12.6% in 2016 to 23.2% this past season. This can potentially be explained by the embrace of analytics by many teams, with many coaches actually utilizing fourth down cheat sheets. Whatever the exact reasons, the way the game is approached on fourth has fundamentally changed.

### Aggressiveness by Distance Over Time
We can drill down on the overall aggressiveness trend by viewing fourth down attempts by distance. Distance from the line to gain can be categorized into 3 different buckets; short (1-3 yards), medium (4-6 yards) and long (7+ yards).

In [8]:
#| echo: false

# Create a column for short, medium, and long
data = data.assign(distance = np.where(
    (data["ydstogo"] <= 3), "short", np.where(
        (data["ydstogo"] > 3) & (data["ydstogo"] <= 6), "medium", np.where(
            (data["ydstogo"] > 6), "long", data["ydstogo"]
        )
    )
)
)

In [9]:
#| echo: false

# Get each seasons attempts and non attempts
dis_rates = data.groupby(["season", "go", "distance"]).agg(
    n = ("go", "count")
).reset_index()

# pivot
dis_rates = pd.pivot(data= dis_rates, index= ["season", "distance"], columns= "go", values = "n" ).reset_index().rename(columns ={0.0: "no_go", 1.0: "go"} ).rename_axis(columns=None)
# pivot works by indexing on the column you want to represent an individual row, then column repressing the column you want to have a new column created for each distinct value in it, and values is what should go into that new column

dis_rates = dis_rates.assign(go_rate = round(dis_rates["go"] / (dis_rates["go"] + dis_rates["no_go"]), 4))


In [10]:
#| echo: false

distance_chart = (alt.Chart(data= dis_rates, title="Fourth Down Aggressiveness by Distance (2016-2025)")
        .mark_line(
            point=alt.OverlayMarkDef(size=50, color = "black", opacity=0.5)
            )
        .encode(
            x=alt.X("season:N", title='Season', axis = alt.Axis(labelAngle=0)),
            y=alt.Y("go_rate:Q", title= "Go Rate"),
            tooltip=["season", "go_rate", "distance"],
            color= alt.Color('distance:N', scale= alt.Scale(range=colors)).legend(orient="right")
        )
        .properties(width=600, height=300)
    ).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)


distance_chart

alt.Chart(...)

The majority of the increase in overall fourth down attempt rate has been driven by attempts in short yardage situations as it increased from 29% to more than **56%** in 2025. Teams on fourth and short are more likely than not to go for it. The rate of growth for medium and long is much more muted, however they still have both  increased over the same time span. Medium has gone from 9% to 18%, and long has gone from 5% to 7%. These trends make sense, as teams are more confident they can convert a fourth down with less yards to go, which incentives their aggressiveness in those situations.

### Postseason vs Regular Season
Another potential factor that could influence the fourth down decision making process is if the game is occurring in the postseason or not. Games played during the postseason carry an inherently higher level of urgency - a team has no margin of error to lose. Therefore, teams will be more aggressive and/or desperate.

In [11]:
#| echo: false

pos_reg = data.groupby(["season_type", "go"]).agg(
    n = ("go", "count")
).reset_index()

pos_reg = pos_reg.pivot(index= ["season_type"], columns= "go", values= "n").reset_index().rename_axis(columns= None).rename(columns ={0.0: "no_go", 1.0: "go"})

pos_reg = pos_reg.assign(go_rate = round(pos_reg["go"] / (pos_reg["go"] + pos_reg["no_go"]), 4))

In [12]:
#| echo: false

(alt.Chart(data= pos_reg, title="Aggressiveness by Season Type 2016-2025").mark_bar().encode(
            x=alt.X("season_type:N", title='Season Type', sort = "-y",axis = alt.Axis(labelAngle=0)),
            y=alt.Y("go_rate:Q", title="Go Rate"),
            tooltip=['go_rate', "season_type"],
            color = alt.Color("season_type:N" ,scale = alt.Scale(range  = colors))
        ).properties(width=350, height=450).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )

alt.Chart(...)

Fourth downs in the postseason did feature a higher go rate when compared to the regular season (20.5% vs 17.7%), supporting the idea that teams are more incentivuzed to go for it when their season is on the line.

### Attempts by Quarter

Now pivoting to another way to view the state a game is in is by quarter. NFL games are played with a *finite* amount of time, 4 quarters each containing 15 minute (and an additional overtime quarter if regulation ends tied). The thought process here is that as time progresses during a game, the urgency that teams play with is increased, and they will be more aggressive.

In [13]:
#| echo: false

quarter = data.groupby(["qtr", "go"]).agg(
    n = ("go", "count")
).reset_index()

quarter = quarter.pivot(index= ["qtr"], columns= "go", values= "n").reset_index().rename_axis(columns= None).rename(columns ={0.0: "no_go", 1.0: "go"})

quarter = quarter.assign(go_rate = round(quarter["go"] / (quarter["go"] + quarter["no_go"]), 4))



In [14]:
#| echo: false

(alt.Chart(data= quarter, title="4th Down Aggressiveness by Quarter (2016-2025)").mark_bar().encode(
            x=alt.X("qtr:N", title='Quarter', sort = "x",axis = alt.Axis(labelAngle=0)),
            y=alt.Y("go_rate:Q", title="Go Rate"),
            tooltip=['go_rate', "qtr"],
            color = alt.when(qtr=4).then(alt.value("#CCBB44")).otherwise(alt.value("#4477AA"))
        ).properties(width=600, height=450).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )

alt.Chart(...)

As the quarter of the game increases, the go rate foes as well, with the highest far and away coming in the fourth(30%), which is double the rate of the next (regulation) quarter. Teams are definitely more and more aggressive as time progresses. The one exception to this is the "5th" quarter (overtime) in which teams seem to have a partial reset of their state of aggressiveness. The go rate in overtime (18%) is far lower than the 4th, however it is still higher than the go rate and the third quarter (14%).

### Attempts by Time Remaining
Still viewing a game through the lens of time, we will examine the distribution of time remaining in the game for the sample of all fourth downs where teams went for it. If the findings from aggressiveness by quarter are to hold up, we would expect most attempts to come with a lower amount of seconds remaining.

In [15]:
#| echo: false

fourth_att = data[data["go"] == 1]
time_left = fourth_att[["game_seconds_remaining"]]

In [16]:
#| echo: false

alt.Chart(data= time_left,  title="4th Down Attempt Distribution by Time Remaining").mark_bar(color= "#66CCEE").encode(
    x = alt.X("game_seconds_remaining:Q",title= "Seconds Remaining in Game", bin=True),
    y = alt.Y("count():Q", title= "Fourth Down Attempts"),
    tooltip= ["count()"]
).properties(width=600, height=450).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)

alt.Chart(...)

The vast majority of fourth down attempts occurred between 0-500 seconds remaining in the game (0-8 minutes). *Time* evidently does have a significant influence on if a team will go for it or not on fourth down.

### Attempts by Point Differential 
Finally we will examine the distribution of point differential. Traditional football intuition would dictate that teams ahead (i.e. positive point differential) should be more conservative while teams that are behind (negative point differential) will be more aggressive, due to the pressure of trailing. 

In [17]:
#| echo: false

score_diff = fourth_att[["score_differential"]]

alt.Chart(data= score_diff,  title="4th Down Attempt Distribution by Score Differential").mark_bar(color= "#66CCEE").encode(
    x = alt.X("score_differential:Q",title= "Score Differential", bin=True),
    y = alt.Y("count():Q", title= "Fourth Down Attempts"),
    tooltip= ["count()"]
).properties(width=600, height=450).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)

alt.Chart(...)

The most fourth down attempts are located in the range of score differential of -10:0. The complement to this range, 0 to 10, had a significant but not as high number of attempts. This supports the idea that teams who are trailing are more willing to go for it than teams that are leading. When looking at situations with a much higher differential, wither positive or negative, we observe a far lower number of attempts. As the absolute value of score differential increases, fourth down attempts go down.

### Team Aggressiveness Rankings
Now lets see how the 4th down aggressiveness of each NFL team from 2016-2025 varies.

In [18]:
#| echo: false

# Get each seasons attempts and non attempts
team_rates = data.groupby(["go", "posteam"]).agg(
    n = ("go", "count")
).reset_index()

# pivot
team_rates = pd.pivot(data= team_rates, index = "posteam", columns= "go", values = "n").reset_index().rename(columns ={0.0: "no_go", 1.0: "go"})

# calculate team rates
team_rates = team_rates.assign(go_rate = round(team_rates["go"] / (team_rates["go"] + team_rates["no_go"]), 4))

# rename posteam col
team_rates = team_rates.rename(columns= {"posteam": "team"})

# pivot works by indexing on the column you want to represent an individual row, then column repressing the column you want to have a new column created for each distinct value in it, and values is what should go into that new column
team_rates = team_rates.sort_values(by = "go_rate", ascending=False)


In [19]:
#| echo: false

(alt.Chart(data= team_rates, title="4th Down Aggressiveness by Team 2016-2025").mark_bar().encode(
            x=alt.X("go_rate:Q", title='Go Rate'),
            y=alt.Y("team:N", title="Team", sort = '-x'),
            tooltip=['go_rate', "team"],
            color = alt.when(team="PHI").then(alt.value("#CCBB44")).when(team="SEA").then(alt.value("#EE6677")).otherwise(alt.value("#4477AA"))
        ).properties(width=600, height=600).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )

alt.Chart(...)

The Philadelphia Eagles have been the most aggressive team over this 10 year time span, attempting fourth downs at a rate of 22.5%. On the flip side, the least aggressive team was the Seattle Seahawks. The intriguing thing about this, is these are the two previous winners of the most recent superbowl, and each clearly had a distinct fourth down philosophy from each other. This highlights the fact that how aggressive a team is on fourth does not necessarily make or break their overall success, it is about having a philosophy and sticking to it.

## 4. Success rate {#sec-success}
The most objective way to measure fourth down performance is by what percent of the time you convert. 
`Success Rate (sr) = (Successful Fourth Down Conversions) / (Total Fourth Down Conversion Attempts)`

### Overall league success rate 2016-2025
To generate some context to better understand how certain teams measure in success rate, we will first examine the leagues overall success rate on fourth down.

In [20]:
#| echo: false
# filter for just plays where go is equal to 1
fourth_att = data[data["go"] == 1]
sr = fourth_att.groupby("fourth_down_converted").agg(
    count = ("fourth_down_converted", "size")
).reset_index()

sr = pd.pivot_table(data= sr, index= None, columns= "fourth_down_converted", values= "count").rename_axis(columns= None).rename(
    columns= {0.0: "fail",
              1.0 : "success"}
)
sr["sr"] = sr['success'] /(sr['success'] + sr['fail'])

sr # the leagues overall success rate on 4th downs from 2016-2025 was 53%

,fail,success,sr
count,3290.0,3746.0,0.532405


The overall success rate of fourth down attempts the previous 10 years is 53%, so more or less a coin flip. Teams that are able to perform better than this gain a significant competitive advantage.

### Overall Team success rates 2016-2025
Lets see how teams measure up in terms of success rate to each other.

In [21]:
#| echo: false
sr_team = fourth_att.groupby(["fourth_down_converted", "posteam"]).agg(
    count = ("fourth_down_converted", "size")
).reset_index()

sr_team = sr_team.pivot(index="posteam", columns="fourth_down_converted", values="count").rename_axis(columns=None).reset_index().rename(columns={0.0: "fail",
                                                                                                                                        1.0: "success"}).rename(columns={"posteam": "team"})
sr_team["n"] = (sr_team['success'] + sr_team['fail'])
sr_team["sr"] = (sr_team['success'] /(sr_team['success'] + sr_team['fail']))

sr_team = sr_team.sort_values(by  = "sr",ascending=False)

In [22]:
#| echo: false

(alt.Chart(data= sr_team, title="4th Down SR by Team 2016-2025").mark_bar().encode(
            x=alt.X("sr:Q", title='Success Rate'),
            y=alt.Y("team:N", title="Team", sort = '-x'),
            tooltip=['sr', "team"],
            color = alt.when(team="KC").then(alt.value("#CCBB44")).when(team="NYJ").then(alt.value("#EE6677")).otherwise(alt.value("#4477AA"))
        ).properties(width=600, height=600).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )

alt.Chart(...)

The Chiefs are a *significant* outlier here, they are the only team with a success rate north of 60%, and they succeed 2/3 of the time. The Chiefs having Patrick Mahomes, the best QB in the league likely heavily contributes to their efficiency. This success has translated into tangible team success as well, the chiefs have won 3 superbowls since 2016, the most by any team in that span. The New York Jets, one of the least successful teams in the NFL, have the worst fourth down success rate, a putrid 45.6% success rate, and no playoff appearances to show for it.

### Teams by success rate and Aggressiveness
Now we combine the team rankings of success rate and attempt rate into one succinct visualization. A vertical line goes through the mean fourth down attempt rate of all 32 teams, and a horizontal line goes though the overall league success rate.

In [23]:
#| echo: false

# merge the success and go rate dfs together
team_quadrants = team_rates.merge(sr_team, how = "left", on = "team")[["team", "go_rate", "sr"]]

quad = alt.Chart(data=team_quadrants, title="Team SR and Aggressiveness (2016-2025)").mark_circle(
    size=60, color='red'
).encode(
    x=alt.X('go_rate', title="Go Rate", scale=alt.Scale(zero=False)),
    y=alt.Y('sr', title="Success Rate", scale=alt.Scale(zero=False)),
    tooltip=['go_rate', 'sr', 'team']
)

hline = alt.Chart().mark_rule(color='blue', strokeDash=[4, 4]).encode(
    y=alt.YDatum(sr["sr"].iloc[0])
)

vline = alt.Chart().mark_rule(color='blue', strokeDash=[4, 4]).encode(
    x=alt.XDatum(team_quadrants["go_rate"].mean())
)

(quad + hline + vline).properties(
    width=600, height=500
).configure_axis(
    titleFontSize=15, labelFontSize=12
).configure_title(fontSize=25)

alt.LayerChart(...)

We can interpret the quadrants as the following:

- quadrant 1: successful and aggressive
    - These teams take full advantage of fourth down.
- quadrant 2: successful and conservative
  - These teams are efficient on fourth down, but are more selective about when they go for it (Chiefs ar the outlier at the top).
- quadrant 3: unsuccessful and conservative
  - These teams are inefficient and selective about when they go for it.
- quadrant 4: unsuccessful and aggressive
    These teams are inefficient and aggressive about when they go (Jets are located at the lowest point). 

## 5. Event Analysis {#sec-event}
We now pivot to viewing fourth down attempts from how the offense approached the down - either passing or rushing. Much of the data used in this section is participation data, which simply means someone watching the film of each play labeled who was on the field, what formation the offense and defense were in, what route was run by receivers, etc. One caveat to this however, is that on plays where a rush occurred, the defense's coverage was unable to be recorded. Furthermore, coverage type data only has been tracked since 2018.

### Run vs Pass Overview
Game context often dictates when a team attempts a pass or rush. Passes tend to gain more yards than rushes on average, so when teams are farther away from the line to gain they tend to pass, while when they are close to the line to gain they tend to run. Therefore, pass and run plays need to be evaluated separately for the purposes of fourth down analysis, the situation surrounding each is inherently different.

In [24]:
#| echo: false

sr_rp = fourth_att.groupby(["play_type","fourth_down_converted"]).agg(
    count = ("fourth_down_converted", "size")
).reset_index()

sr_rp = pd.pivot_table(data= sr_rp, index= ["play_type"], columns= "fourth_down_converted", values= "count").rename_axis(columns= None).rename(
    columns= {0.0: "fail",
              1.0 : "success"}
).reset_index()

sr_rp["sr"] = sr_rp['success'] /(sr_rp['success'] + sr_rp['fail'])

# Create Dataframe to show mean yardage
rp_yds = fourth_att.groupby("play_type").agg(
    mean_ydstogo = ("ydstogo", "mean")
).reset_index()

# Merge together
sr_rp = sr_rp.merge(rp_yds, how = "left", on = "play_type")
sr_rp
# Maybe co

,play_type,fail,success,sr,mean_ydstogo
0,pass,2470.0,1964.0,0.442941,5.465719
1,run,820.0,1782.0,0.684858,1.747886


Looking at the above table, passes only have a 44% sr when compared to rushes (68%). However, looking at the average yards to go, pass attempts on fourth occurred almost four yards farther way on average in comparison than runs, so this discrepancy makes sense.

In [25]:
#| echo: false

# Defining fourth down datasets for either a run or a pass.

runs = fourth_att[fourth_att["play_type"] == "run"]
passes = fourth_att[fourth_att["play_type"] == "pass"]

# Function for getting sr based on run or pass column
def rp_col(play_type, column):
    """ Get SR on 4th down for a column describing a run or pass
    """
    # Case for if you want to look at run or pass
    if play_type == "run":
        play_data = fourth_att[fourth_att["play_type"] == "run"]
    else: 
        play_data = fourth_att[fourth_att["play_type"] == "pass"]

    # group by desired column and if 4th down was converted
    col_data = play_data.groupby([f"{column}","fourth_down_converted"]).agg(
        count = ("fourth_down_converted", "size")
    ).reset_index()

    # Pivot data to make plotting possible
    col_data = col_data.pivot(index= f"{column}", columns= "fourth_down_converted", values= "count").reset_index().rename_axis(columns= None).rename(
        columns= {0.0: "fail",
                1.0 : "success"}
    )

    # Create success rate column
    col_data["sr"] = (col_data['success'] /(col_data['success'] + col_data['fail'])).round(3)
    
    # Return data and sort by sr
    return col_data.sort_values(by = "sr", ascending =False).reset_index(drop=True)

### Runs

#### Run SR by run gap
On run plays, the rusher has three gap types they can try to penetrate:

- End: Running outside the tackles
- Tackle: running between the tackle and the guard
- Guard: running between the guard and the center

In [26]:
#| echo: false

run_gap = rp_col(play_type= "run", column= "run_gap")


(alt.Chart(data= run_gap, title="Run SR by Gap").mark_bar().encode(
            x=alt.X("run_gap:N", title='Run Gap', sort = "-y",axis = alt.Axis(labelAngle=0)),
            y=alt.Y("sr:Q", title="Success Rate"),
            tooltip=['sr', "run_gap"],
            color = alt.Color("run_gap:N" ,scale = alt.Scale(range  = colors))
        ).properties(width=500, height=500).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )

alt.Chart(...)

The gap with the highest success rate was end (71.6%), which indicates that the kind of runs on fourth down which are most successful are ones that attack the edge, rather than running it up the gut. Runs in this gap were 4% more successful than runs in the guard gap, and 5% more successful than runs in the tackle gap.

#### Run SR by run location
Run location is quite intuitive, it simply denotes the direction in which the run went.

In [27]:
#| echo: false

run_location = rp_col(play_type= "run", column= "run_location")

(alt.Chart(data= run_location, title="Run SR by Run Location").mark_bar().encode(
            x=alt.X("run_location:N", title='Run Location', sort = "-y",axis = alt.Axis(labelAngle=0)),
            y=alt.Y("sr:Q", title="Success Rate"),
            tooltip=['sr', "run_location"],
            color = alt.Color("run_location:N" ,scale = alt.Scale(range  = colors))
        ).properties(width=500, height=500).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )

alt.Chart(...)

The location of the run was relatively stable across all three gaps, as runs in the middle, left, and right were 70%, 69%, and 68% respectively.

#### Run SR by offensive formation
The offensive formation a team is in can dictate how the defense plays them, with formations that have the QB lined up under center suggesting a likely run, while formations with the qb in some form of shotgun indicating a likely run.

In [28]:
#| echo: false

run_off_form = rp_col(play_type= "run", column= "offense_formation")

(alt.Chart(data= run_off_form, title="Run SR by Offensive Formation").mark_bar().encode(
            x=alt.X("offense_formation:N", title='Run Location', sort = "-y",axis = alt.Axis(labelAngle=0)),
            y=alt.Y("sr:Q", title="Success Rate"),
            tooltip=['sr', "offense_formation"],
            color = alt.Color("offense_formation:N" ,scale = alt.Scale(range  = colors))
        ).properties(width=600, height=500).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )


alt.Chart(...)

Interestingly the formation with the highest success rate was empty, with 75% of their attempts resulting in a conversion. Empty means their is no running back for the QB to hand it off too, it is instead a designed QB run. QB runs are effective, but it is important to remember that QBs are the most valuable position on a team, and should not have their chance of inure risked with a high usage rate on fourth down.

### Passes

#### Pass SR by coverage type (man/zone)
Defensive coverages can either have the classification of man or zone. Man means that each receiver will have a defender assigned to guard them specifically, and zone means that each defender is assigned to a specific part of the *field* to guard.

In [29]:
#| echo: false

man_zone = rp_col(play_type= "pass", column= "defense_man_zone_type")

(alt.Chart(data= man_zone, title="Pass SR by Offensive Formation").mark_bar().encode(
            x=alt.X("defense_man_zone_type:N", title='Man Zone Type', sort = "-y",axis = alt.Axis(labelAngle=0)),
            y=alt.Y("sr:Q", title="Success Rate"),
            tooltip=['sr', "defense_man_zone_type"],
            color = alt.Color("defense_man_zone_type:N" ,scale = alt.Scale(range  = colors))
        ).properties(width=600, height=500).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )


alt.Chart(...)

Offenses found the most success converting fourth downs on passes when teams were in Man, with a SR of ~50%, where fourth downs with a zone coverage had a 41.5% SR. One potential issue hear is that some bias for yards to go is being encoded here, as when a team passes in short situations defenses are more likely to be in man coverage, while long situations will likely result in a defense lining up in zone. 

#### Pass SR by coverage defensive formation
Now we drill down even further, examining the specific type of man or zone coverage that defenses used on pass plays.

In [30]:
#| echo: false

cov_type = rp_col(play_type= "pass", column = "defense_coverage_type")

(alt.Chart(data= cov_type[cov_type["defense_coverage_type"] != "BLOWN"], title="Pass SR by Defensive Coverage").mark_bar().encode(
            x=alt.X("defense_coverage_type:N", title='Coverage Type', sort = "-y",axis = alt.Axis(labelAngle=0)),
            y=alt.Y("sr:Q", title="Success Rate"),
            tooltip=['sr', "defense_coverage_type"],
            color = alt.Color("defense_coverage_type:N" ,scale = alt.Scale(range  = colors_2))
        ).properties(width=700, height=500).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )

alt.Chart(...)

When defenses line up in cover 0, offenses have the highest SR out of any coverages, 51.6%. Defenses like lining up in cover 0 due to the fact that it is able to get a high rate of pressure onto the QB, however in doing so they leave less defenders guarding the pass, making them more prone to give up a first down.

#### Pass SR by offensive formation
Now we view fourth down pass attempts by offensive formation like we previously did for runs.

In [31]:
#| echo: false

pass_off_form = rp_col(play_type= "pass", column= "offense_formation")

(alt.Chart(data= pass_off_form, title="Pass SR by Offensive Formation").mark_bar().encode(
            x=alt.X("offense_formation:N", title='Offensive Formation', sort = "-y",axis = alt.Axis(labelAngle=0)),
            y=alt.Y("sr:Q", title="Success Rate"),
            tooltip=['sr', "offense_formation"],
            color = alt.Color("offense_formation:N" ,scale = alt.Scale(range  = colors_2))
        ).properties(width=700, height=500).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )

alt.Chart(...)

What is counter intuitive here is that jumbo formation, a type of formation typically reserved for runs, has the highest SR, at 71.4%. What is likely happening here is that defenses expect a team to run, however because the offense goes with a pass, the defense is deceived and is much less effective.

#### Pass SR by route run
The route ran column denotes the route of the receiver who was targeted on the pass.

In [32]:
#| echo: false

route = rp_col(play_type= "pass", column= "route")

(alt.Chart(data= route[route["fail"] > 9], title="Pass SR by Route Run").mark_bar().encode(
            y=alt.Y("route:N", title='Route Run', sort = "-x",axis = alt.Axis(labelAngle=0)),
            x=alt.X("sr:Q", title="Success Rate"),
            tooltip=['sr', "route"],
            color = alt.Color("route:N" ,scale = alt.Scale(range  = colors_2))
        ).properties(width=500, height=500).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )

alt.Chart(...)

Quick outs and slants had an extremely high success rate, nearly 60%, however routes such as go, post, and corner all had success rates below 43% (go as low as 33%). What is probably happening here is that quick outs and slants are used to convert shorter yardage pass situations, while corners, posts, and gos are used in farther pass situations, hence the large gap in SR between them.

## 6. Modeling 4th Down Attempts {#sec-modeling}
Now we will model fourth down attempts. Our target variable (what we are predicting) is if a team went for it or not, and it will be a function of where the team with possession has the ball (yardline 1-99), seconds left in the game, yards to gain, and point differential. These are the variables which based on the previous analysis, should be the strongest indicators of if a team goes for it or not. All that is left now is to run and evaluate the model. 

### Modeling and Evaluation
We will reserve a holdout test set of the data for evaluation (measured by accuracy) of 20%, and we will train a Gradient Boosting Classifier model with a learning rate of 30% to predict fourth down attempts.

In [33]:
# Define X and y variables
X = data[["yardline_100", "game_seconds_remaining", "ydstogo", "score_differential"]]
y = data["go"]

# Partition X and y into training and test data using 20% for holdout
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state = 42)

# Fit gradient boosting model
gb = GradientBoostingClassifier(learning_rate= 0.3)
gb.fit(X_train, y_train)



print(f'Train Set Accuracy: {round(gb.score(X_train, y_train), 3)}')

print(f'Test Set Accuracy: {round(gb.score(X_test, y_test), 3)}')

Train Set Accuracy: 0.925
Test Set Accuracy: 0.919


When evaluated on the test set, the model was able to correctly predict ~92% of fourth down attempts correctly, as either attempts or non attempts. This is a very promising performance, and more tinkering with hyperparameter may even yield better results.

### Feature Importance
Using the calculated feature importance from the gradient boosting classifier, we can quantify how important each variable used to predict if a team went for it or not really was to the model.

In [34]:
#| echo: false

importances = pd.DataFrame({
    "col" : gb.feature_names_in_,
    "importance": gb.feature_importances_
}).sort_values(by = "importance", ascending = False).reset_index(drop= True)

(alt.Chart(data= importances, title="Feature Importance of GB Model").mark_bar().encode(
            x=alt.X("col:N", title='Predictor', sort = "-y",axis = alt.Axis(labelAngle=0)),
            y=alt.Y("importance:Q", title="Importance"),
            tooltip=['importance', "col"],
            color = alt.Color("col:N" ,scale = alt.Scale(range  = colors))
        ).properties(width=700, height=500).configure_axis(titleFontSize=15, labelFontSize= 12).configure_title(fontSize=25)
    )

alt.Chart(...)

From the model's perspective, the most important variable in determining if a team will go for it or not on fourth down is yards to go (i.e distance), followed by score differential, then what yardline the offense has the ball on, and how much time remaining in the game there is. All 4 of these variables make up a sizeable chunk of the feature importance, indicating that they all do meaningfully contribute to predicting fourth down attempts.

## 7. Discussion
This analysis mainly focused on identifying the conditions surrounding fourth down attempts and what drives their success. This is more of an "after the fact" view of things. The next step from here would be to try and model the interaction that advanced metrics such as win probabilities and EPA have with going for it on fourth down, as a true competitive advantage can be gained from knowing when going for it will increase your chance of *winning*. 

## 8. Conclusion
As previously stated, the goal of this investigation was to determine the factors at play that determine when NFL teams go for it on fourth down, and what determines if that attempt was successful or not. Yards to go (distance), score differential, yardline, and time remaining were the variables that had the most influence on if a team would go for it or not. In terms of what drives fourth down success, the type of coverage (man or zone) and route run by targeted receiver heavily influenced the success of passes, while the offensive formation and run gap influenced the success of run plays. 